# TP4 :  Low-budget  Learning

# Introduction

**Context :**

Assume we are in a context where few "gold" labeled data are available for training, say 

$$\mathcal{X}_{\text{train}} = \{(x_n,y_n)\}_{n\leq N_{\text{train}}}$$

where $N_{\text{train}}$ is small. Hence we are not in the context of classical well-organized benchmark datasets.

To make your life easier, you also get a validation set $\mathcal{X}_{\text{val}}$ representative of the test set, but you cannot use it for training. You can only use it to select the best model and hyperparameters.

A large test set $\mathcal{X}_{\text{test}}$ exists but is not accessible. We also assume that we have a limited computational budget (e.g., limited GPU access).


In this practical, we will use the `resnet10` architecture. 

# QUESTIONS

## Grading

You will need to provide 4 files : 
1. This Notebook
2. `utils.py`
3. `drawing_lora.png`
4. `cutmix.png`

Some of the code will be automatically graded so please follow the instructions carefully.

You will also need to participate in a competition on https://www.codabench.org to get your model evaluated on the hidden test set (the link to the competition will be provided on the course forum). Before submitting anything make sure to read the instructions on the competition page. The evaluation metric is the accuracy. However, *this is not a competition* as it is not necessary to get the best score to get a good grade. If you get a good score, and follow the instructions, you will get a good grade.

`utils.py` will be imported during the testing phase so please make sure:
- it does not execute any code when imported
- it does not depend on any module that is not standard (e.g., not `torch`, `torchvision`, `numpy`, etc.)


General instructions:
- Please provide clear and short answers between `<div class="alert alert-info">  <your answer>  </div>` tags (when it's not code).
- For each question that involves training a model:
    - Give the number of trained parameters.
    - You must provide the training curves (train & validation accuracy/loss vs epochs) in the notebook.
    - You must explain the choices you made (hyperparameters, etc). (A short justification is enough. For instance, "I used default hyperparameters." does not need further explanation. Or "I tried (0.1,0.01,0.001) and picked 0.01 because it gave the best validation accuracy." is enough.)
    - You must comment on the accuracy obtained.
- If you use a seed for reproducibility, please make sure it is a personal one using something like `hash("your_firstname_your_lastname")`.

<div class="alert alert-info">  Example of answer  </div>

In [ ]:
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn
from torchmetrics.classification import ConfusionMatrix, Accuracy
from torchvision.transforms import v2
import torchvision.transforms as T
from torch.utils.data import default_collate

import random

import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model
import copy

from tqdm.auto import  tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import os

if not os.path.exists("data"):
    os.mkdir("data")
if not os.path.exists("data/TP4_images"):
    os.mkdir("data/TP4_images")
if not os.path.exists("data/TP4_images/north_dataset_test"):
    !cd data/TP4_images && wget -O north_dataset_train.zip  "https://nextcloud.lisn.upsaclay.fr/index.php/s/yzQRWE2YjmFn9WA/download/north_dataset_train.zip" && unzip north_dataset_train.zip
    !cd data/TP4_images && wget -O north_dataset_test.zip  "https://nextcloud.lisn.upsaclay.fr/index.php/s/zntidWrFdYsGMDm/download/north_dataset_test.zip" && unzip north_dataset_test.zip
dir_path = "data/TP4_images/"

In [ ]:
val_dataset = datasets.ImageFolder(
    "data/TP4_images/north_dataset_test",
    transform=transforms.Compose([transforms.ToTensor()]),
)

train_dataset = datasets.ImageFolder(
    "data/TP4_images/north_dataset_sample",
    transform=transforms.Compose([transforms.ToTensor()]),
)


metric = ConfusionMatrix(task="multiclass", num_classes=2).to(device)


def model_instancier(**kwargs):
    """
    Instanciate a ResNet10 model (ResNet18 with only 1 block per layer).

    Parameters
    ----------
    **kwargs: dict
        Keyword arguments to pass to the ResNet18 constructor.

    Returns
    -------
    model: nn.Module
        The instantiated ResNet10 model.
    """
    _model = models.resnet18(**kwargs)
    for i in range(1, 5):
        setattr(_model, f"layer{i}", getattr(_model, f"layer{i}")[0])
    return _model


base_model = model_instancier()
classifier_name = "fc"


print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

In [ ]:
def plot_confmat(confmat):
    plt.imshow(confmat.cpu(), cmap="Blues")
    plt.xticks([0,1], ["Class 0","Class 1"])
    plt.yticks([0,1], ["Class 0","Class 1"])
    for i in range(2):
        for j in range(2):
            plt.text(j,i,int(confmat[i,j]), ha="center", va="center", color="white" if confmat[i,j]>confmat.max()/2 else "black")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Question 1: 
>  Create `last_layer.py` to change the last layer of the resnet model so that it fits the problem.

<div class="alert alert-info">  We choose to use two layers because it is the minimum required for non-linearity and is sufficient to fit the data. </div>

In [ ]:
from collections import OrderedDict

class LastLayer(nn.Module):
    
    def __init__(self):
        super(LastLayer, self).__init__()
        self.out_features = 2
        self.fc = nn.Sequential(OrderedDict([
            ('fc_Linear1', nn.Linear(512, 8)),
            ('activation', nn.ReLU()),
            ('fc_Linear2', nn.Linear(8, self.out_features))
        ]))
        
    def forward(self, x):
        return self.fc(x)

In [ ]:
from last_layer import LastLayer

setattr(base_model, classifier_name, LastLayer())
assert (
    getattr(base_model, classifier_name).out_features == 2
)  # we could also change the last layer to have 1 output. Do it with 2 so that it matches our tests procedure during grading.

## Question 2: 
> Provide a function `precompute_features` in `utils.py` that creates a new dataset from the features precomputed by the model.

In [ ]:
def precompute_features(
    model: models.ResNet, dataset: torch.utils.data.Dataset, device: torch.device
) -> torch.utils.data.Dataset:
    """
    Create a new dataset with the features precomputed by the model.

    If the model is $f \circ g$ where $f$ is the last layer and $g$ is
    the rest of the model, it is not necessary to recompute $g(x)$ at
    each epoch as $g$ is fixed. Hence you can precompute $g(x)$ and
    create a new dataset
    $\mathcal{X}_{\text{train}}' = \{(g(x_n),y_n)\}_{n\leq N_{\text{train}}}$

    Arguments:
    ----------
    model: models.ResNet
        The model used to precompute the features
    dataset: torch.utils.data.Dataset
        The dataset to precompute the features from
    device: torch.device
        The device to use for the computation

    Returns:
    --------
    torch.utils.data.Dataset
        The new dataset with the features precomputed
    """
    
    model.eval()
    model.to(device)
    
    fc = getattr(model, classifier_name)
    setattr(model, classifier_name, nn.Identity())
    
    features = []
    labels = []
    
    with torch.no_grad():
        for x, y in dataset:
            feat = model(x.unsqueeze(0).to(device)).cpu()
            features.append(feat)
            labels.append(y)
    
    features = torch.cat(features, dim=0)
    labels = torch.tensor(labels)
    
    setattr(model, classifier_name, fc)
    
    return torch.utils.data.TensorDataset(features, labels)

## Question 3: 
> Train the last layer of a randomly initialized resnet model.  Provide the training process in the notebook with training curve. Comment on the accuracy. 

<div class="alert alert-info">  Here we used generic hyperparameters. Tuning them would be useless because we are training on random features, which makes it impossible to learn from the data. The last Layer has 4122 parameters.  </div>

In [ ]:
from utils import precompute_features

In [ ]:
base_model = model_instancier()
setattr(base_model, classifier_name, LastLayer())
base_model = base_model.to(device)

train_features_dataset = precompute_features(base_model, train_dataset, device=device)
val_features_dataset = precompute_features(base_model, val_dataset, device=device)

In [ ]:
batch_size = 6
lr = 1e-2
weight_decay = 0

train_loss = []
val_loss = []

train_acc = []
val_acc = []

fc = getattr(base_model, classifier_name)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fc.parameters(), lr=lr, weight_decay=weight_decay)

train_dataloader = torch.utils.data.DataLoader(train_features_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_features_dataset, batch_size=6)

print(f'number of parameters : {count_parameters(fc)}')

In [ ]:
def validate(model, val_dataloader, criterion):
    
    model.eval()
    epoch_loss = []
    correct_val = 0
    total_val = 0
    cm_metric = ConfusionMatrix(task="multiclass", num_classes=2).to(device)
    acc_metrc = Accuracy(task="multiclass", num_classes=2).to(device)
    with torch.no_grad():
        for features, labels in val_dataloader: 
            features = features.to(device)
            labels = labels.to(device) 
            y = model(features)
            loss = criterion(y, labels)
            epoch_loss.append(loss.cpu().item())
            
            predicted = y.argmax(dim=1)
            correct_val += (predicted == labels).sum().cpu().item()
            total_val += len(labels)
            
            preds = torch.argmax(y, dim=1)
            cm_metric.update(preds, labels)
            acc_metrc.update(preds, labels)
    
    confmat = cm_metric.compute().cpu()
    acc = acc_metrc.compute().cpu().item()
    
    cm_metric.reset()
    acc_metrc.reset()
    
    mean_loss = sum(epoch_loss)/len(epoch_loss)
    
    return mean_loss, acc, confmat
    

def train(epochs,
          model,
          optimizer,
          criterion,
          train_dataloader,
          val_dataloader,
          train_loss=None,
          train_acc=None,
          val_loss=None,
          val_acc=None, 
          scheduler=None):
    
    if None in [train_loss, train_acc, val_loss, val_acc]:
        train_loss, train_acc, val_loss, val_acc = [], [], [], []
    
    mean_loss, acc, _ = validate(model, val_dataloader, criterion)
    
    best_model = copy.deepcopy(model)
    best_acc = acc
    
    for _ in tqdm(range(epochs)):
        model.train()
        epoch_loss = []
        correct_train = 0
        total_train = 0
        
        for features, labels in train_dataloader:
            features = features.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            
            y = model(features)
            loss = criterion(y, labels)
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            
            epoch_loss.append(loss.cpu().item())
            
            predicted = y.argmax(dim=1)
            if len(labels.shape) == 2:
                labels = labels.argmax(dim=1)
            correct_train += (predicted == labels).sum().cpu().item()
            total_train += len(labels)
        
        train_loss.append(sum(epoch_loss)/len(epoch_loss))
        train_acc.append(correct_train / total_train)

        mean_loss, acc, _ = validate(model, val_dataloader, criterion)
        
        if acc > best_acc:
            best_model = copy.deepcopy(model)
            best_acc = acc
        
        val_loss.append(mean_loss)
        val_acc.append(acc)
    
    return best_model, train_loss, val_loss, train_acc, val_acc

In [ ]:
epochs = 100

_ , train_loss, val_loss, train_acc, val_acc = train(epochs, fc, optimizer, criterion, train_dataloader, val_dataloader)

plt.subplot(2, 1, 1)
plt.plot(train_loss, label='train loss')
plt.plot(val_loss, label='val loss')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(train_acc, label='train accuracy')
plt.plot(val_acc, label='val accuracy')
plt.legend()
plt.show()

<div class="alert alert-info"> The model achieve 75% on the train set because it could overfit some of the noise. On the validation dataset we get 50% as expected of random features as input. </div>

## Question 4:
>  Now start from a pretrained model on imagenet (https://pytorch.org/vision/stable/models.html#) and only train the last layer. Provide the training process in the notebook with training curve. 

This is the first question evaluated on the codabench platform.

<div class="alert alert-info"> We use a small batch size of 4 due to the limited size of the training dataset. We evaluated learning rates ranging from 1e-1 to 1e-4, with 1e-2 yielding the best performance (accuracy). Weight decay is applied to stabilize training and mitigate overfitting, which is particularly likely with such a small dataset. Values between 1e-3 and 1e-6 were tested, and 1e-4 was selected. </div>

In [ ]:
base_model = model_instancier(weights="DEFAULT")
setattr(base_model, classifier_name, LastLayer())
base_model = base_model.to(device)

In [ ]:
train_features_dataset = precompute_features(base_model, train_dataset, device=device)
val_features_dataset = precompute_features(base_model, val_dataset, device=device)

In [ ]:
batch_size = 4
lr = 1e-2
weight_decay = 1e-4

train_loss = []
val_loss = []

train_acc = []
val_acc = []

fc = getattr(base_model, classifier_name)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(fc.parameters(), lr=lr, weight_decay=weight_decay)

train_dataloader = torch.utils.data.DataLoader(train_features_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_features_dataset, batch_size=6)

print(f'number of parameters : {count_parameters(fc)}')

In [ ]:
epochs = 100

best_model , train_loss, val_loss, train_acc, val_acc = train(epochs, fc, optimizer, criterion, train_dataloader, val_dataloader,
                                                              train_loss, train_acc, val_loss, val_acc)

plt.subplot(2, 1, 1)
plt.plot(train_loss, label='train loss')
plt.plot(val_loss, label='val loss')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(train_acc, label='train accuracy')
plt.plot(val_acc, label='val accuracy')
plt.legend()
plt.show()

In [ ]:
_, acc, confmat = validate(fc, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

In [ ]:
_, acc, confmat = validate(best_model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

<div class="alert alert-info">  The performance consistently reaches an accuracy of 0.88, and the model obtained at the end of training corresponds to the best performance observed throughout training. This suggests that no overfitting is present.  </div>

### Save your model

In [ ]:
# Save the last layer weights for grading on codabench
torch.save(best_model.to('cpu').state_dict(), "last_layer_finetune.pth")
best_model.to(device)

### Check that you can load your model

In [ ]:
model = model_instancier(weights="DEFAULT")
fc = LastLayer()
fc.load_state_dict(
    torch.load("last_layer_finetune.pth", weights_only=True, map_location=device)
)
setattr(model, classifier_name, fc)
model = model.to(device)
model.eval()

_, acc, confmat = validate(fc, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

## Question 5:
> Perform  LoRA https://arxiv.org/pdf/2106.09685  on the model  ( We are perfectly fine if you use external library for this question only ) 

Intermediate question : Describe LoRA and the difference with LoRA for Convolutions in this paper : https://arxiv.org/pdf/2401.17868  in a few lines  (you do not have to implement ConvLora)

<div class="alert alert-info">  LoRA is applied to fine-tune linear layers. For a given linear layer with a weight matrix of size (m, n), the idea is to learn a low-rank matrix of the same size and add it to the original weight. We obtain this low-rank matrix by multiplying two matrices A (m, r) and B (r, n), where r is small (less than m and n).

During training, the base network is frozen because it is pretrained, and we do not want to modify its weights directly. Instead, we only learn the parameters in A and B. The final weights are therefore W + AB, where W are the original pretrained weights. At the start of the training A and B are set to 0 to save the original capabilities of the network.

LoRA for convolution extends this idea by inserting a convolution layer between A and B. If x is the input, we compute A(conv(Bx)) instead of ABx.
  </div>

<div class="alert alert-info">  We use the peft package to fine tune with LoRA.
  </div>

In [ ]:
def lora(model, r, alpha):
    config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=["conv1", "conv2"],
        lora_dropout=0.1,
    )

    model = get_peft_model(model, config)
    
    for params in model.fc.parameters():
        params.requires_grad = True
    
    model.print_trainable_parameters()
    
    return model

def warmup(warmup_steps):

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0
    
    return lr_lambda

<div class="alert alert-info">  We started with an r and alpha of 4, then tested r values of 1, 2, and 3, and found that r = 3 worked best (accuracy). We increased the batch size from 4 to 8 to speed up training. We decreased the learning rate to 1e-4, as it gave the best accuracy and stable training. Given that the training set is still very small and we now have 50,259 parameters, we used a weight decay of 1e-3 to avoid overfitting and stabilize training. </div>

In [ ]:
lora_model = copy.deepcopy(model)
lora_model = lora_model.to(device)


r = 3
alpha = 4

batch_size = 8
lr = 1e-4
weight_decay = 1e-3

lora_model = lora(lora_model, r, alpha)

train_loss = []
val_loss = []

train_acc = []
val_acc = []

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=lr, weight_decay=weight_decay)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=6)

In [ ]:
epochs = 100

best_model , train_loss, val_loss, train_acc, val_acc = train(epochs, lora_model, optimizer, criterion, train_dataloader, val_dataloader,
                                                     train_loss, train_acc, val_loss, val_acc)

plt.subplot(2, 1, 1)
plt.plot(train_loss, label='train loss')
plt.plot(val_loss, label='val loss')
plt.yscale('log')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(train_acc, label='train accuracy')
plt.plot(val_acc, label='val accuracy')
plt.legend()
plt.show()

In [ ]:
_, acc, confmat = validate(lora_model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

In [ ]:
_, acc, confmat = validate(best_model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

<div class="alert alert-info">
  The best model obtained throughout the training usually achieves the same 0.88 accuracy as the last-layer fine-tune. However, the model obtained at the last iteration is not consistently as good. This shows that training is much less stable with 10x more parameters.
</div>

## Question 6:
> Let's do some Data Augmentation https://en.wikipedia.org/wiki/Data_augmentation . Load some alteration of the data from the `torchvision.transforms` module and incorporate them into your training pipeline.

Intermediate question : Check CutMix  (https://pytorch.org/vision/stable/auto_examples/transforms/plot_cutmix_mixup.html#sphx-glr-auto-examples-transforms-plot-cutmix-mixup-py) and explain it with a small drawing. 

You can find many other data augmentation techniques here: https://docs.pytorch.org/vision/0.15/transforms.html

This is the second question evaluated on the codabench platform.

<div class="alert alert-info">
  CutMix is a data augmentation technique where patches from one image are cut and pasted onto another image, and the labels are mixed proportionally to the area of the patches. It helps regularize training, reduce overfitting, and often improves accuracy on small datasets.
</div>

<div class="alert alert-info">
  We keep the same values of r, alpha, and batch size as before. Because the data is much more diverse with data augmentation, we don't need as much regularization (decrease weight decay to 1e-4), and we can increase the learning rate to 1e-3. We still have 50259 parameters for this model.
</div>

In [ ]:
lora_model = copy.deepcopy(model)
lora_model = lora_model.to(device)


r = 3
alpha = 4

batch_size = 8
lr = 1e-3
weight_decay = 1e-4

lora_model = lora(lora_model, r, alpha)

train_loss = []
val_loss = []

train_acc = []
val_acc = []

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=lr, weight_decay=weight_decay)

<div class="alert alert-info">
  We use random horizontal and vertical flips with crop-and-resize data augmentation to maximize diversity. We also apply CutMix. These data augmentation techniques are applied randomly so that some training samples remain in the same distribution as the original dataset.
</div>

In [ ]:
classic_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.2),
    T.RandomVerticalFlip(p=0.2),
    T.RandomApply(
        [v2.RandomResizedCrop((256, 256), scale=(0.6, 1))],
        p=0.2
    )
])

cutmix = v2.CutMix(num_classes=2)
CUTMIX_PROB = 0.5

def collate_fn(batch):
    images, labels = default_collate(batch)
    
    images = torch.stack([classic_transforms(img) for img in images])
    
    if random.random() < CUTMIX_PROB:
        images, labels = cutmix(images, labels)
        
    return images, labels

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=6)

In [ ]:
epochs = 1000

best_model, train_loss, val_loss, train_acc, val_acc = train(epochs, lora_model, optimizer, criterion, train_dataloader, val_dataloader,
                                                     train_loss, train_acc, val_loss, val_acc)

plt.subplot(2, 1, 1)
plt.plot(train_loss, label='train loss')
plt.plot(val_loss, label='val loss')
plt.yscale('log')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(train_acc, label='train accuracy')
plt.plot(val_acc, label='val accuracy')
plt.legend()
plt.show()

In [ ]:
_, acc, confmat = validate(lora_model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

In [ ]:
_, acc, confmat = validate(best_model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

### Save your model

In [ ]:
# Merge LoRA weights back into the base model
merged_model = best_model.merge_and_unload()  # TODO: Merge LoRA weights back into the base model
assert isinstance(merged_model, models.ResNet)

merged_model = merged_model.to('cpu')
torch.save(merged_model.state_dict(), "final_model.pth")

### Check that you can load your model

In [ ]:
from last_layer import LastLayer

classifier_name = "fc"
device = torch.device("cpu")
model = model_instancier()
setattr(model, classifier_name, LastLayer())

model.load_state_dict(
    torch.load("final_model.pth", weights_only=True, map_location=device)
)
model.eval()

In [ ]:
_, acc, confmat = validate(model, val_dataloader, criterion)

print(f'accuracy = {acc}')
plot_confmat(confmat)

<div class="alert alert-info">
  The training is very unstable, but we can still observe some improvement by keeping the best model obtained throughout training based on the accuracy metric on the validation dataset. This best model consistently reaches > 0.9 accuracy, but it is difficult to determine whether this improvement is genuine or simply due to chance, since such a model could appear randomly across many epochs. Without a test set, we did not find a way to differentiate between these two possibilities.
</div>

# Some advice

In our experiments, we only used SGD and a laptop GPU. We recommend not hesitating to use a large number of epochs (e.g., 100, 200, etc.). We did not use any learning rate scheduler but you can try if you want. Many data augmentation techniques exist, you can try them and see if they improve your performance. You can also try to combine them. For instance, you can try to combine CutMix with some geometric transformations (e.g., random crop, random horizontal flip, etc.).

The improvement from LoRA and data augmentation is quite hard to see on the small validation set. If you get even a small improvement on the validation set, it is likely that you will get a better score on the test set, except if you overfit the validation set. With an honest improvement on the validation set, you should be able to get a good grade.